# Salmon quantification of bulk RNA-seq

Builds a Salmon index of the Ensembl mouse transcriptome (GRCm39 cDNA, release 110), quantifies every
sample (selective alignment, automatic library type, GC-bias correction) and sums the estimated reads
of each gene's transcripts into a gene-level count matrix (as tximport with `countsFromAbundance = "no"`),
the format of the matrix deposited in GEO (GSE338230). Runs on Linux, for example Google Colab.

**Input:** a folder of adapter- and quality-trimmed FASTQ files, `<sample>` + `R1` (and `R2` for
paired-end) suffixes as set below.

**Output** (in `OUT_DIR`): the Salmon index, one Salmon folder per sample (`quant/<sample>/quant.sf`) and
`counts_gene.csv` (Ensembl gene IDs x samples).

In [ ]:
"""Install Salmon (Linux binary)."""
import glob, subprocess

SALMON_VERSION = "1.10.0"
subprocess.run(f"wget -q -nc https://github.com/COMBINE-lab/salmon/releases/download/v{SALMON_VERSION}/"
               f"salmon-{SALMON_VERSION}_linux_x86_64.tar.gz && tar xzf salmon-{SALMON_VERSION}_linux_x86_64.tar.gz",
               shell=True, check=True)
SALMON = glob.glob(f"salmon-{SALMON_VERSION}*/bin/salmon")[0]
subprocess.run([SALMON, "--version"], check=True)

In [ ]:
"""Settings."""
from pathlib import Path

FASTQ_DIR = Path("trimmed")        # trimmed FASTQ files
OUT_DIR = Path("results/salmon")
R1, R2 = "_trimmed_R1.fastq.gz", "_trimmed_R2.fastq.gz"   # a sample without an R2 file is run single-end
THREADS = 12

In [ ]:
"""Parameters (values used in the paper)."""
CFG = dict(
    transcriptome="https://ftp.ensembl.org/pub/release-110/fasta/mus_musculus/cdna/Mus_musculus.GRCm39.cdna.all.fa.gz",
    gtf="https://ftp.ensembl.org/pub/release-109/gtf/mus_musculus/Mus_musculus.GRCm39.109.gtf.gz",  # transcript -> gene
    quant_args=["-l", "A", "--validateMappings", "--gcBias"],
)

In [ ]:
"""Imports and versions."""
import gzip, re, sys, urllib.request
import pandas as pd

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | salmon {SALMON_VERSION}")


def fetch(url):
    """Download a reference file once into OUT_DIR."""
    path = OUT_DIR / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

## Index and quantification

In [ ]:
"""Salmon index (k = 31) and quantification of every sample."""
index = OUT_DIR / "salmon_index"
if not (index / "versionInfo.json").exists():
    subprocess.run([SALMON, "index", "-p", str(THREADS), "-t", str(fetch(CFG["transcriptome"])), "-i", str(index)],
                   check=True)

samples = sorted(p.name[: -len(R1)] for p in FASTQ_DIR.glob(f"*{R1}"))
for s in samples:
    out = OUT_DIR / "quant" / s
    if (out / "quant.sf").exists():
        continue
    r1, r2 = FASTQ_DIR / f"{s}{R1}", FASTQ_DIR / f"{s}{R2}"
    reads = ["-1", str(r1), "-2", str(r2)] if r2.exists() else ["-r", str(r1)]
    subprocess.run([SALMON, "quant", "-p", str(THREADS), "-i", str(index), *CFG["quant_args"], *reads,
                    "-o", str(out)], check=True)
print(f"{len(samples)} samples quantified")

## Gene-level counts

Transcript versions are dropped and the estimated reads (`NumReads`) of the transcripts of each gene are
summed, using the transcript-to-gene map of the Ensembl GTF in `CFG["gtf"]`; transcripts absent from the
GTF are left out.

In [ ]:
"""Gene-level count matrix."""
tx2gene = {}
with gzip.open(fetch(CFG["gtf"]), "rt") as fh:
    for line in fh:
        f = line.split("\t", 8)
        if len(f) == 9 and f[2] == "transcript":
            tx = re.search(r'transcript_id "([^"]+)"', f[8]).group(1)
            tx2gene[tx] = re.search(r'gene_id "([^"]+)"', f[8]).group(1)
tx2gene = pd.Series(tx2gene)

counts = {}
for s in samples:
    q = pd.read_csv(OUT_DIR / "quant" / s / "quant.sf", sep="\t", index_col=0)["NumReads"]
    q.index = q.index.str.replace(r"\.\d+$", "", regex=True)
    q = q[q.index.isin(tx2gene.index)]
    counts[s] = q.groupby(tx2gene[q.index].values).sum()
counts = pd.DataFrame(counts).fillna(0).rename_axis("gene_id")
counts.round(3).to_csv(OUT_DIR / "counts_gene.csv")
print(f"{counts.shape[0]} genes x {counts.shape[1]} samples")